# Entrenamiento: Predictor de compatibilidad de amistad

Entrena y evalúa `CompatibilityNet` (red neuronal 5→8→4→1 en PyTorch) con el dataset sintético de 8000 pares, ejecutando los scripts del repositorio.

**Cómo usarlo:** ejecutá las celdas en orden (Entorno de ejecución → Ejecutar todas). La red es chica y corre bien en CPU; la GPU es opcional (Entorno de ejecución → Cambiar tipo de entorno de ejecución).

Repositorio: https://github.com/RocioArruabarrena/RedNeuronal_Colab

## 1. Clonar el repositorio
Clona el repo en la rama indicada en `BRANCH`, o lo actualiza si ya existe.

In [ ]:
import os

REPO_URL = "https://github.com/RocioArruabarrena/RedNeuronal_Colab.git"
BRANCH = "main"
REPO_DIR = "/content/RedNeuronal_Colab"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

## 2. Instalar dependencias
Instala `requirements.txt` sin torch, numpy ni pandas, que ya vienen con Colab.

In [ ]:
# torch, numpy y pandas ya vienen en Colab: no los reinstalo para no romper el entorno
!grep -viE "^(torch|numpy|pandas)==" requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt

## 3. Dataset y split
Muestra las columnas de los CSV sintéticos y divide los datos en train / validación / test (70 / 15 / 15, estratificado). El split se guarda en `data/splits.npz`.

In [ ]:
!head -1 profiles_unificado.csv
!head -1 friendships_unificado.csv
!python scripts/split_dataset_csv.py

## 4. Entrenamiento
Entrena con Adam, BCELoss y early stopping (paciencia de 20 épocas sobre `val_loss`). Guarda los pesos en `models/compatibility_net.pt`.

In [ ]:
!python scripts/entrenar_red.py

## 5. Evaluación
Evalúa el modelo en el conjunto de test (umbral 0.5). Accuracy esperada: alrededor de 0.81.

In [ ]:
!python scripts/evaluate_model.py

## 6. Búsqueda de arquitectura
Genera `sintetico_features.csv` (no se versiona en el repo) y compara 10 configuraciones y una regresión logística de referencia con validación cruzada de 3 folds. Todas dan un AUC de alrededor de 0.84, así que se mantiene la arquitectura 5→8→4→1. El resultado queda en `results/`.

In [ ]:
!python scripts/exportar_sintetico_csv.py
%run scripts/colab_busqueda_arquitectura.py

## 7. Descargar los resultados
Descarga los pesos entrenados, el split y el CSV de la búsqueda de arquitectura.

In [ ]:
import os
from google.colab import files

archivos = [
    "models/compatibility_net.pt",
    "data/splits.npz",
    "results/resultados_arquitectura_sintetico_features.csv",
]

for ruta in archivos:
    if os.path.isfile(ruta):
        print(f"OK  {ruta}  ({os.path.getsize(ruta)} bytes)")
        files.download(ruta)
    else:
        print(f"NO EXISTE: {ruta}")